### ЗАДАЧА: Резервирование товаров по складам

Команда закупок получает пакет заявок на резерв товаров по складам.
Нужно собрать систему, которая:
- принимает корректные заявки,
- отбрасывает неправильные или невыполнимые запросы,
- обновляет остатки на складе после успешного резерва,
- собирает отдельный журнал ошибок,
- позволяет быстро понять, кто зарезервировал больше всего товара и на каком складе осталось меньше всего позиций.

В части строк указаны неизвестные склады или товары,
в части количество заполнено неверно,
а некоторые заявки пытаются зарезервировать больше доступного остатка или дублируют уже обработанный `request_id`.


In [ ]:
from dataclasses import dataclass
from typing import Optional


stocks = {
    'MSK-1': {'keyboard': 10, 'mouse': 20, 'monitor': 4},
    'SPB-2': {'keyboard': 6, 'dock': 5, 'monitor': 2},
    'KZN-3': {'mouse': 7, 'dock': 3, 'laptop': 2},
}

# rows: request_id|client|warehouse_id|sku|quantity
rows = [
    'RQ-100|Acme|MSK-1|keyboard|3',
    'RQ-101|Beta|SPB-2|dock|2',
    'RQ-102|Acme|MSK-1|monitor|5',
    'RQ-103|Delta|X-999|mouse|1',
    'RQ-104|Gamma|KZN-3|laptop|0',
    'RQ-105|Beta|SPB-2|chair|1',
    'RQ-101|Beta|MSK-1|mouse|4',
    'RQ-106|Acme|MSK-1|mouse|7',
    'RQ-107|Kira|KZN-3|laptop|1',
]


class ReservationError(Exception):
    pass


class RowFormatError(ReservationError):
    pass


class WarehouseNotFoundError(ReservationError):
    pass


class ProductNotFoundError(ReservationError):
    pass


class QuantityError(ReservationError):
    pass


class StockLimitError(ReservationError):
    pass


class DuplicateRequestError(ReservationError):
    pass


@dataclass(order=True)
class ReservationRequest:
    request_id: str
    client: str
    warehouse_id: str
    sku: str
    quantity: int


class Warehouse:
    def __init__(self, warehouse_id, products):
        # TODO: сохранить warehouse_id
        self.warehouse_id = warehouse_id

        # TODO: создать отдельную копию словаря products
        self.products = products.copy()

        # TODO: создать список reservations
        self.reservations: list[ReservationRequest] = []

    def has_sku(self, sku):
        # TODO: вернуть True/False, есть ли такой sku в self.products
        return sku in self.products

    def available(self, sku):
        # TODO: вернуть текущий остаток по sku
        if self.has_sku(sku):
            return self.products[sku]
        return 0

    def reserve(self, request):
        # TODO: если request.sku отсутствует -> raise ProductNotFoundError(...)
        if not self.has_sku(request.sku):
            raise ProductNotFoundError(f"")
        
        # TODO: если request.quantity > available(...) -> raise StockLimitError(...)
        if request.quantity > self.available(request.sku):
            raise StockLimitError(f"")

        # TODO: уменьшить остаток на складе
        self.products[request.sku] -= request.quantity

        # TODO: добавить request в self.reservations
        self.reservations.append(request)

    def total_left(self):
        # TODO: вернуть сумму всех остатков на складе
        return sum(self.products.values())

    def reserved_total(self):
        # TODO: вернуть сумму quantity по всем self.reservations
        return sum()


class ReservationService:
    def __init__(self, stocks):
        # TODO: создать warehouses вида warehouse_id -> Warehouse(...)
        self.warehouses: dict[str, WarehouseNotFoundError]

        # TODO: создать списки accepted и errors
        self.accepted: list[ReservationRequest] = []
        self.errors: list[tuple[str, str, str]] = []

        # TODO: создать множество processed_ids
        pass

    def parse_request(self, row):
        # TODO: split по '|'
        parts = row.split("|")

        # TODO: ожидать 5 частей: request_id, client, warehouse_id, sku, quantity_raw
        request_id, client, warehouse_id, sku, quantity_raw = parts

        # TODO: quantity_raw преобразовать в int
        quantity_raw = int(quantity_raw)

        # TODO: если warehouse_id не существует -> WarehouseNotFoundError
        if warehouse_id not in self.warehouses:
            raise WarehouseNotFoundError(f"{warehouse_id} не найдена")
        
        # TODO: если quantity <= 0 -> QuantityError
        # TODO: вернуть объект ReservationRequest(...)
        return ReservationRequest(request_id, client, warehouse_id, sku, quantity_raw)


    def submit(self, row):
        # TODO: внутри try вызвать parse_request(row)
        try:
            request = self.parse_request(row)

        # TODO: если request.request_id уже в processed_ids -> DuplicateRequestError


        # TODO: затем warehouses[request.warehouse_id].reserve(request)
            self.warehouses[request.warehouse_id].reserve(request)

        # TODO: после успеха добавить request_id в processed_ids
            self.processed_ids.append(request_id)

        # TODO: добавить request в self.accepted
            self.accepted.append(request)

        # TODO: ReservationError сохранить в self.errors как (row, error_type, message)
        except ReservationError as e:
            self.errors.append((row, type(e).__name__, e))


    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)


    def client_totals(self):
        # TODO: собрать dict вида client -> total_reserved_quantity
        pass

    def top_client(self):
        # TODO: использовать client_totals()

        # TODO: вернуть tuple(client, total_quantity) с максимумом
        pass

    def lowest_stock_warehouse(self):
        # TODO: найти склад с минимумом total_left()

        # TODO: вернуть tuple(warehouse_id, total_left)
        

    def warehouse_snapshot(self):
        # TODO: собрать dict вида warehouse_id -> копия текущих остатков products
        pass

    def find_request(self, request_id: str) -> Optional[ReservationRequest]:
        # TODO: вернуть Optional[ReservationRequest]
        # TODO: пройтись по self.accepted и найти нужную заявку

        # TODO: если не найдено -> вернуть None
        return None


service = ReservationService(stocks)

# TODO: загрузить rows через service.load(rows)
service.load(rows)

# TODO: вывести принятые заявки
# TODO: вывести ошибки
# TODO: вывести warehouse_snapshot()
# TODO: вывести client_totals()
# TODO: вывести top_client()
# TODO: вывести lowest_stock_warehouse()
# TODO: вывести find_request('RQ-107')
